In [2]:
import pandas as pd

data_quality=pd.read_csv("C:/Rohit/Virtual_Gigafactory/raw_data/quality.csv")
data_production=pd.read_csv("C:/Rohit/Virtual_Gigafactory/raw_data/production_b001.csv")


merged_data=pd.merge(data_production,data_quality,on='Cell_ID')
print(merged_data.columns)



Index(['Cell_ID', 'Timestamp', 'Machine_x', 'Shift', 'Operator', 'Recipe',
       'Batch_ID', 'Cell_Type', 'Voltage', 'Current', 'Temperature',
       'Machine_y', 'OCV', 'DCIR', 'Capacity', 'Final_Result', 'Defect_Type'],
      dtype='object')


In [6]:
print(merged_data.shape)
print(merged_data.groupby('Final_Result')['Temperature'].mean())

mean_by_voltage=data_production.groupby('Machine')["Voltage"].agg(["mean","std"])
print(mean_by_voltage)


(500, 17)
Final_Result
FAIL    27.545000
PASS    28.614567
Name: Temperature, dtype: float64
             mean       std
Machine                    
M1       3.678245  0.031467
M2       3.678413  0.029303
M3       3.681067  0.029779
M4       3.677105  0.022770
M5       3.681431  0.028540
M6       3.679086  0.036732
M7       3.675906  0.029935
M8       3.681603  0.029373


In [8]:
mean_by_voltage.sort_values(by='std')

,mean,std
Machine,,
M4,3.677105,0.022770
M5,3.681431,0.028540
M2,3.678413,0.029303
M8,3.681603,0.029373
M3,3.681067,0.029779
M7,3.675906,0.029935
M1,3.678245,0.031467
M6,3.679086,0.036732


In [33]:
print(data_production.groupby('Machine')['Temperature'].mean().sort_values(ascending=False))

Machine
M7    31.098413
M8    28.490882
M5    28.431538
M2    28.338254
M3    28.263333
M1    28.176415
M4    28.161053
M6    27.889429
Name: Temperature, dtype: float64


In [38]:
numeric_data=data_production.select_dtypes(include='number')
report=numeric_data.groupby(data_production['Machine']).mean().sort_values(by='Temperature',ascending=False)

In [61]:
grouped_by_temp=data_production.groupby('Machine')['Temperature'].mean()
# print(grouped_by_temp)
overall_abnormal_threshold=grouped_by_temp.mean()+grouped_by_temp.std()

unusual_machine=grouped_by_temp[grouped_by_temp>overall_abnormal_threshold]

print(unusual_machine)



Machine
M7    31.098413
Name: Temperature, dtype: float64


In [54]:
type(grouped_by_temp.items())


zip

In [69]:
average_temp=data_production.groupby('Machine')[['Temperature']].mean()

average_temp['Average']=average_temp['Temperature'].mean()
average_temp['Difference']=average_temp['Temperature']-average_temp['Average']
print(average_temp)


         Temperature    Average  Difference
Machine                                    
M1         28.176415  28.606165   -0.429750
M2         28.338254  28.606165   -0.267911
M3         28.263333  28.606165   -0.342831
M4         28.161053  28.606165   -0.445112
M5         28.431538  28.606165   -0.174626
M6         27.889429  28.606165   -0.716736
M7         31.098413  28.606165    2.492248
M8         28.490882  28.606165   -0.115282


In [73]:
full_summary=data_production.groupby('Machine')[['Voltage','Current','Temperature']].agg({'Voltage':'mean',
                                                                                          'Current':'mean',
                                                                                          'Temperature':['mean','std']})
print(full_summary)

          Voltage    Current Temperature          
             mean       mean        mean       std
Machine                                           
M1       3.678245  48.378868   28.176415  1.809220
M2       3.678413  48.357619   28.338254  1.690083
M3       3.681067  48.040667   28.263333  2.134376
M4       3.677105  48.307719   28.161053  1.950246
M5       3.681431  48.341385   28.431538  1.858493
M6       3.679086  48.260857   27.889429  1.804355
M7       3.675906  48.345156   31.098413  1.682261
M8       3.681603  47.493382   28.490882  2.070090


In [76]:
print(data_production.groupby('Machine')['Voltage'].count())

Machine
M1    53
M2    63
M3    60
M4    57
M5    65
M6    70
M7    64
M8    68
Name: Voltage, dtype: int64


In [89]:
print(data_quality.columns)
print(data_quality.loc[1:3,:])

Index(['Cell_ID', 'Machine', 'OCV', 'DCIR', 'Capacity', 'Final_Result',
       'Defect_Type'],
      dtype='object')
   Cell_ID Machine    OCV   DCIR  Capacity Final_Result  Defect_Type
1   100001      M4  3.666  17.05     27.53         PASS          NaN
2   100002      M2  3.695  16.59     27.69         PASS          NaN
3   100003      M1  3.668  17.18     27.52         PASS          NaN


In [100]:

data_quality['Passed_cells']=data_quality['Final_Result'].map({'PASS':1,"FAIL":0})
final_summary=data_quality.groupby('Machine').agg({'OCV':'mean',
                                                   'DCIR':'mean',
                                                   'Final_Result':'count',
                                                   'Passed_cells':'sum'})

final_summary['Yield']=(final_summary['Passed_cells']/final_summary['Final_Result'])*100



In [99]:
final_summary=final_summary.rename(columns={'Final_Result':'Total_count'})
print(final_summary)

              OCV       DCIR  Total_count  Passed_cells       Yield
Machine                                                            
M1       3.681874  18.067869          183           183  100.000000
M2       3.678206  18.064233          189           189  100.000000
M3       3.678000  18.112434          189           189  100.000000
M4       3.677754  17.842982          171           171  100.000000
M5       3.679483  17.933506          174           172   98.850575
M6       3.678224  17.955381          210           210  100.000000
M7       3.678041  17.854301          193           193  100.000000
M8       3.681932  17.961152          191           191  100.000000


In [5]:
print(data_production.groupby(['Machine','Shift'])['Voltage'].mean().reset_index())

   Machine  Shift   Voltage
0       M1    Day  3.679290
1       M1  Night  3.676773
2       M2    Day  3.682606
3       M2  Night  3.673800
4       M3    Day  3.682200
5       M3  Night  3.679933
6       M4    Day  3.680278
7       M4  Night  3.671667
8       M5    Day  3.676243
9       M5  Night  3.688286
10      M6    Day  3.679744
11      M6  Night  3.678258
12      M7    Day  3.680515
13      M7  Night  3.671000
14      M8    Day  3.681880
15      M8  Night  3.681442


In [7]:
machine_shift_grouped=data_production.groupby(['Machine','Shift'])['Temperature'].agg(['mean','std'])

machine_shift_grouped.sort_values(by='std',ascending=False,inplace=True)

print(machine_shift_grouped.head(3))




                    mean       std
Machine Shift                     
M3      Night  28.244667  2.351599
M8      Night  28.532093  2.303715
M6      Day    28.224615  2.027638


In [20]:
temp_mean_by_machine=data_production.groupby('Machine')['Temperature'].mean()
plant_mean_temp=data_production['Temperature'].mean()
machine_above_plant_mean=temp_mean_by_machine[temp_mean_by_machine>=plant_mean_temp].reset_index()
print(machine_above_plant_mean)
for index,row in machine_above_plant_mean.iterrows():
    print(f'The machine {row['Machine']} having temp {row['Temperature']:.2f} is above plant mean temperature by {(row['Temperature']-plant_mean_temp):.2f}')

  Machine  Temperature
0      M7    31.098413
The machine M7 having temp 31.10 is above plant mean temperature by 2.49


In [32]:
machine_temp_summary=data_production.groupby('Machine')['Temperature'].agg(['mean','std'])
suspected_machines=machine_temp_summary[machine_temp_summary['std']>2.0].reset_index()

if suspected_machines.empty:
    print('No Suspected machines')
else:
    for index,row in suspected_machines.iterrows():
        print(f"The Machine {row['Machine']} needs inspection")
        print(f"Standard deviation: {row['std']}")
        print(f"Mean Temperature: {row['mean']}\n")

The Machine M3 needs inspection
Standard deviation: 2.1343761575507036
Mean Temperature: 28.263333333333332

The Machine M8 needs inspection
Standard deviation: 2.0700895757488933
Mean Temperature: 28.490882352941178



In [51]:
temperature_summary=data_production.groupby('Machine')['Temperature'].agg(['mean','std','max','min']).reset_index()

temperature_summary['Temperature Range']=temperature_summary['max']-temperature_summary['min']

temperature_summary.sort_values(by='std',ascending=False,inplace=True)
print(temperature_summary)

  Machine       mean       std    max    min  Temperature Range
2      M3  28.263333  2.134376  33.16  23.40               9.76
7      M8  28.490882  2.070090  33.15  23.84               9.31
3      M4  28.161053  1.950246  32.03  24.10               7.93
4      M5  28.431538  1.858493  33.12  23.60               9.52
0      M1  28.176415  1.809220  33.12  24.39               8.73
5      M6  27.889429  1.804355  32.81  22.21              10.60
1      M2  28.338254  1.690083  32.15  24.83               7.32
6      M7  31.098413  1.682261  35.12  27.97               7.15


In [59]:
## ticket0013
shift_summary=data_production.groupby('Shift').agg(Average_Temperature=('Temperature','mean'),
                                                   Average_Current=('Current','mean'),
                                                   Average_Voltage=('Voltage','mean'),
                                                   Cell_Count=('Machine','count'))

shift_summary.sort_values(by='Average_Temperature',ascending=False,inplace=True)
                                                                                         
print(shift_summary)


       Average_Temperature  Average_Current  Average_Voltage  Cell_Count
Shift                                                                   
Night            28.686186        48.035339         3.677996         236
Day              28.542167        48.311894         3.680208         264


In [75]:
##ticket0014
high_temp_data=data_production[data_production['Temperature']>25]

high_temp_summary=high_temp_data.groupby('Machine').size().reset_index()
high_temp_summary.rename(columns={'0':'Count'})
# high_temp_summary.sort_values(by='Count',ascending=False, inplace=True)
if high_temp_summary.empty:
    print('No machine with high temp cells')
else:
    print(high_temp_summary)



  Machine   0
0      M1  52
1      M2  62
2      M3  54
3      M4  53
4      M5  64
5      M6  67
6      M7  63
7      M8  63
